# syft-ingest API Exploration

## 1. Environment check

BRIGHTDATA_API_TOKEN : 7a5...1d5


## 2. Core imports

In [2]:
import syft_ingest as si
from syft_ingest import FetchConfig, async_gather, gather

## 3. Simplified gather() API — YouTube video

In [3]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=zY2dAK-pMPI&t=11s"]
)  # Andrew Trask: talk

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

2026-04-13 20:44:53.843 | DEBUG    | syft_ingest.sources.youtube:__init__:83 - YtDlpFetcher initialized with config: {'socket_timeout': 30, 'playlistend': 50, 'download_full_video': False}
2026-04-13 20:44:53.844 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher YtDlpFetcher for youtube/yt-dlp
2026-04-13 20:44:53.844 | DEBUG    | syft_ingest.setup:register_fetchers:28 - Registered YtDlpFetcher for YouTube
2026-04-13 20:44:53.844 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for facebook/brightdata
2026-04-13 20:44:53.844 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for instagram/brightdata
2026-04-13 20:44:53.845 | DEBUG    | syft_ingest.setup:register_fetchers:37 - Registered BrightDataFetcher for Facebook and Instagram
2026-04-13 20:44:53.845 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher LocalFetcher for local/local
2

Items fetched: 1
  Title: Decentralized AI From Scratch (Part 1: My First P2P AI)
  Author: Andrew Trask


In [5]:
corpus.export("./data/corpus.jsonl")

2026-04-13 20:45:08.066 | INFO     | syft_ingest.core.exporters:_export_jsonl:74 - Exported 1 items to data/corpus.jsonl


## 4. Channel enumeration with config

In [6]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@iamtrask"],
    playlistend=3,  # Cap at 3 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

2026-04-13 20:45:34.084 | INFO     | syft_ingest.sources.youtube:fetch:124 - Fetching 1 YouTube URL(s)
2026-04-13 20:45:34.084 | INFO     | syft_ingest.sources.youtube:fetch:135 - Detected channel/playlist URL: https://www.youtube.com/@iamtrask
2026-04-13 20:45:34.085 | DEBUG    | syft_ingest.sources.youtube:_enumerate_channel:314 - Enumerating channel with limit=3
2026-04-13 20:45:34.839 | INFO     | syft_ingest.sources.youtube:_enumerate_channel:351 - Enumerated 3 videos from channel
2026-04-13 20:45:34.840 | INFO     | syft_ingest.sources.youtube:fetch:142 - Enumerated 3 videos from channel
2026-04-13 20:45:34.840 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_and_captions:404 - Extracting metadata for https://www.youtube.com/watch?v=zY2dAK-pMPI
2026-04-13 20:45:37.546 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_and_captions:497 - Extracted 882 caption segments for en
2026-04-13 20:45:37.547 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_

Videos found: 3
  [2026-04-08 00:00:00+00:00] Decentralized AI From Scratch (Part 1: My First P2P AI)
  [2026-03-15 00:00:00+00:00] jobs which are safe from AI: taste, trust, and rare data
  [2018-06-21 00:00:00+00:00] Programming OpenMined.org - Building Federated Learning (4/4)


In [ ]:
corpus.export("./data/corpus2.jsonl")

## 5. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [7]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


In [8]:
corpus = await async_gather(
    "facebook",
    ["https://www.facebook.com/profile.php?id=61583734012155"],
    author="Syft Influencer Test",
    timeout=120,
    poll_interval=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

2026-04-13 20:47:53.424 | INFO     | syft_ingest.sources.brightdata:fetch_async:116 - Fetching 1 URL(s) for facebook
2026-04-13 20:47:53.425 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:130 - Using platform scraper: facebook
2026-04-13 20:47:53.852 | INFO     | syft_ingest.sources.brightdata:fetch_async:152 - Triggering facebook scrape for https://www.facebook.com/profile.php?id=61583734012155
2026-04-13 20:47:54.310 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:158 - Scrape job created: sd_mny0ug1i2lcbq7s0p8
2026-04-13 20:47:54.310 | INFO     | syft_ingest.sources.brightdata:fetch_async:161 - Polling job sd_mny0ug1i2lcbq7s0p8 with timeout=120s, poll_interval=5s
2026-04-13 20:49:09.262 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:172 - Job sd_mny0ug1i2lcbq7s0p8 completed
2026-04-13 20:49:10.015 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:176 - Fetched 162364 bytes from job
2026-04-13 20:49:10.016 | ERROR    | syft_ingest.sources.brightdata:_p

FetchEmptyResultError: No content items found in facebook response

In [ ]:
from syft_ingest.core.fetcher import FetchRequest
from syft_ingest.sources.brightdata import BrightDataFetcher

fetcher = BrightDataFetcher()
request = FetchRequest(
    platform="facebook",
    urls=["https://www.facebook.com/profile.php?id=61583734012155"],
    config={"timeout": 120, "poll_interval": 5},
)

## 6. Instagram scraping with posts_limit for testing

In [ ]:
corpus = await async_gather(
    "instagram",
    ["https://www.instagram.com/syft_influencer_test/"],
    author="Syft Influencer Test",
    timeout=120,
    poll_interval=5,
    posts_limit=5,  # Limit to 5 posts for quick testing (no posts_limit = all posts)
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

## 7. Error handling and validation

In [11]:
# Invalid platform raises ValueError
try:
    corpus = gather("tiktok", ["https://tiktok.com/user"])
except KeyError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")

2026-04-13 20:52:44.064 | ERROR    | syft_ingest.core.gather:gather:83 - No fetcher registered for tiktok: "No fetcher registered for platform 'tiktok' and extractor 'brightdata'. Register one with register_fetcher()."
2026-04-13 20:52:44.064 | ERROR    | syft_ingest.core.gather:gather:80 - Invalid platform: Platform 'youtube' requires urls list


✅ ValueError caught for unsupported platform: "No fetcher registered for platform 'tiktok' and extractor 'brightdata'. Register one with register_fetcher()."
✅ ValueError caught for missing URLs: Platform 'youtube' requires urls list


## 8. Configuration validation with FetchConfig

In [ ]:
# FetchConfig provides validation and IDE autocomplete
# All these options are validated:

config = FetchConfig(
    # YouTube options
    socket_timeout=60,  # Network timeout (seconds)
    playlistend=10,  # Max videos from channel
    download_full_video=False,  # Enable full video download
    # BrightData options
    timeout=300,  # Scrape job timeout
    poll_interval=2,  # Check frequency
    posts_limit=5,  # Limit posts (for testing)
)

print("✅ FetchConfig created with validation:")
print(f"  socket_timeout: {config.socket_timeout}")
print(f"  posts_limit: {config.posts_limit}")
print()

# Invalid values are caught immediately
try:
    bad = FetchConfig(socket_timeout=-1)  # Must be >= 1
except Exception as e:
    print(f"✅ Validation error (expected): {type(e).__name__}")

In [ ]:
# Simplest: just platform + URLs
corpus = si.gather("youtube", ["https://youtube.com/watch?v=..."])

# With author
corpus = si.gather("youtube", ["https://youtube.com/watch?v=..."], author="Andrej")

# With config options
corpus = si.gather(
    "youtube", ["https://youtube.com/@user"], socket_timeout=60, playlistend=10
)

# With Instagram/Facebook (requires BRIGHTDATA_API_TOKEN)
corpus = si.gather(
    "instagram", ["https://instagram.com/user/"], timeout=120, posts_limit=5
)

# Export results
# corpus.export("jsonl", output="out.jsonl")

print("✅ API ready for use!")